In [2]:
import sys
sys.path.append("..")  # add parent directory to path so we can import project modules

In [3]:
# Imports & DB connection
import pandas as pd
from sqlalchemy import text
from data_collection import engine

In [4]:
# Load base dataset — join witt game logs with pitcher stats and park factors
with engine.connect() as conn:
    df = pd.read_sql(text("""
        SELECT
            w.game_id,
            w.date,
            w.season,
            w.home_away,
            w.tb,
            w.opponent_id,
            w.opponent,
            p.pitcher_name,
            p.throws,
            p.era,
            p.whip,
            p.k_per_9,
            pf.park_name,
            pf.park_factor,
            pf.park_factor_1b,
            pf.park_factor_2b,
            pf.park_factor_3b,
            pf.park_factor_hr
        FROM witt_game_logs w
        JOIN pitcher_game_logs p ON w.game_id = p.game_id
        JOIN park_factors pf ON (
            CASE WHEN w.home_away = 'home' THEN 118
                 ELSE w.opponent_id
            END = pf.team_id
        )
        ORDER BY w.date
    """), conn)

print(df.shape)
print(df.head())

(626, 18)
   game_id        date  season home_away  tb  opponent_id  \
0   662766  2022-04-07    2022      home   2          114   
1   662765  2022-04-09    2022      home   0          114   
2   662755  2022-04-10    2022      home   2          114   
3   662754  2022-04-11    2022      home   0          114   
4   662017  2022-04-12    2022      away   0          138   

              opponent   pitcher_name throws  era  whip  k_per_9  \
0  Cleveland Guardians   Shane Bieber      R  4.2   1.3      8.8   
1  Cleveland Guardians    Zach Plesac      R  4.2   1.3      8.8   
2  Cleveland Guardians  Cal Quantrill      R  4.2   1.3      8.8   
3  Cleveland Guardians   Aaron Civale      R  4.2   1.3      8.8   
4  St. Louis Cardinals  Dakota Hudson      R  4.2   1.3      8.8   

          park_name  park_factor  park_factor_1b  park_factor_2b  \
0  Kauffman Stadium           97             100              99   
1  Kauffman Stadium           97             100              99   
2  Kauffma

In [5]:
# Feature Engineering
df = df.sort_values("date").reset_index(drop=True)

# 1. Rolling TB averages (shift(1) so we never leak the current game)
df["tb_avg_7"]  = df["tb"].shift(1).rolling(7,  min_periods=3).mean()
df["tb_avg_15"] = df["tb"].shift(1).rolling(15, min_periods=7).mean()

# 2. TB in previous game (hot/cold signal)
df["tb_lag1"] = df["tb"].shift(1)

# 3. Encode categoricals
df["is_home"]   = (df["home_away"] == "home").astype(int)
df["pitcher_r"] = (df["throws"] == "R").astype(int)

# 4. Drop rows where rolling features are NaN (early season games)
df_model = df.dropna(subset=["tb_avg_7", "tb_avg_15", "tb_lag1"]).reset_index(drop=True)

print(df_model.shape)
print(df_model[["date", "tb", "tb_lag1", "tb_avg_7", "tb_avg_15", "is_home", "pitcher_r"]].head(10))

(619, 23)
         date  tb  tb_lag1  tb_avg_7  tb_avg_15  is_home  pitcher_r
0  2022-04-16   2      0.0  1.000000   1.000000        1          1
1  2022-04-19   0      2.0  1.000000   1.125000        1          1
2  2022-04-20   0      0.0  1.000000   1.000000        1          1
3  2022-04-21   2      0.0  0.714286   0.900000        1          1
4  2022-04-22   1      2.0  1.000000   1.000000        0          1
5  2022-04-23   2      1.0  1.142857   1.000000        0          1
6  2022-04-24   1      2.0  1.000000   1.076923        0          0
7  2022-04-26   3      1.0  1.142857   1.071429        0          0
8  2022-04-27   1      3.0  1.285714   1.200000        0          1
9  2022-04-28   1      1.0  1.428571   1.133333        0          1
